In [1]:
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr
import pandas as pd
import rpy2.rinterface_lib.sexp as SEXP
from rpy2.robjects.vectors import ListVector
import os
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import friedmanchisquare, wilcoxon

from itertools import combinations

Error importing in API mode: ImportError('On Windows, cffi mode "ANY" is only "ABI".')
Trying to import in ABI mode.
c:\Python312\Lib\site-packages\rpy2\rinterface\__init__.py:1211: UserWarning: Environment variable "PATH" redefined by R and overriding existing variable. Current: "c:\Python312;c:\Users\millenium\AppData\Roaming\Python\Python312\Scripts;C:\Program Files\Microsoft SDKs\Azure\CLI2\wbin;C:\Users\millenium\.jabba\jdk\temurin@23.0.2\bin;E:\Program Files\ImageMagick-7.1.1-Q16-HDRI;C:\Python312\Scripts\;C:\Python312\;c:\program files (x86)\jdk/bin;c:\windows\system32;c:\windows;c:\windows\system32\wbem;c:\windows\system32\windowspowershell\v1.0\;c:\windows\system32\openssh\;c:\program files\nvidia corporation\nvidia nvdlisr;e:\program files\git\cmd;c:\program files\docker\docker\resources\bin;c:\users\millenium\appdata\local\microsoft\windowsapps;c:\program files\autofirma\autofirma;c:\users\millenium\appdata\roaming\nvm;c:\program files\nodejs;c:\program files\dotnet\;e:\mave

In [2]:
parameter_translation_map = {
    "A_viewerServerSelectionDistribution": "SP",
    "B_maxReservation": "MR [B]",
    "B_existingViewerServerSelectionDistribution": "ESP",
    "B_newViewerServerParentSelectionDistribution": "OSP",
    "B_existingViewerServerSelectionTreeTraversal": "EST",
    "B_newViewerServerParentSelectionTreeTraversal": "OST",
    "C_maxReservation": "MR [C]",
    "C_reservationType": "RT",
    "C_alreadyLinkedTreeTraversal": "EVST",
    "C_alreadyLinkedDistribution": "EVSP",
    "C_linkedSpaceTreeTraversal": "OVST",
    "C_linkedSpaceDistribution": "OVSP",
    "C_viewerAndLinkFromDistribution": "NVSP",
    "C_publisherServerSelectionDistribution": "PSP",
    "maxReservation": "MR",
}
distribution_cols = ["SP", "ESP", "OSP", "EVSP", "OVSP", "NVSP", "PSP"]
tree_traversal_cols = ["EST", "OST", "EVST", "OVST"]
distribution_translation_map = {
    "LOW_LOAD": "LL",
    "HIGH_LOAD": "HL",
    "ROUND_ROBIN": "RR",
    "RANDOM": "RND",
}
tree_traversal_translation_map = {
    "TOP_TO_BOTTOM": "TTB",
    "BOTTOM_TO_TOP": "BTT",
    "IGNORE_TREE_LEVEL": "IT",
}
reservation_type_translation_map = {
    "RESERVATION_PER_SERVER": "SER",
    "RESERVATION_PER_SESSION": "SES",
}
alg_a_parameters = ["SP"]
alg_b_parameters = ["MR [B]", "ESP", "OSP", "EST", "OST", "MR"]
alg_c_parameters = ["MR [C]", "RT", "EVST", "EVSP", "OVST", "OVSP", "NVSP", "PSP", "MR"]

In [3]:
def remove_duplicates(configs, training_exps, testing_exps, elites):
    # Some configs are duplicate of others with more experiments in different instances, we want to keep the most informative ones
    duplicate_configs = configs[configs.duplicated(keep=False)]
    # if a row contains a RND parameter, we want to remove those "duplicate" configs from our true duplicates
    duplicate_configs = duplicate_configs[
        ~duplicate_configs.apply(
            lambda row: any(row[col] == "RND" for col in duplicate_configs.columns),
            axis=1,
        )
    ]
    # Group duplicate rows by their values (excluding the index)
    duplicate_groups = {}
    for idx, row in duplicate_configs.iterrows():
        row_key = frozenset(
            (col, str(row[col]) if pd.notna(row[col]) else "NaN")
            for col in duplicate_configs.columns
        )
        duplicate_groups.setdefault(row_key, []).append(idx)

    # Keep only groups with more than one entry
    duplicate_indices_tuples = [
        tuple(indices) for indices in duplicate_groups.values() if len(indices) > 1
    ]

    best_cols = []
    for indices in duplicate_indices_tuples:
        cols = training_exps[list(indices)]
        # Prefer any index present in testing_exps
        if testing_exps.columns.isin(indices).any():
            best_col = next((i for i in indices if i in testing_exps.columns), None)
        else:
            non_na_counts = cols.notna().sum()
            best_col = non_na_counts.idxmax()
        best_cols.append(best_col)
        # Fill gaps in the chosen column from the others
        for other in indices:
            if other == best_col:
                continue
            for row in training_exps.index:
                if pd.isna(training_exps.at[row, best_col]) and pd.notna(training_exps.at[row, other]):
                    training_exps.at[row, best_col] = training_exps.at[row, other]

    # Update elites: replace any dropped duplicate in elites with its kept best_col
    updated_elites = list(elites)
    for indices, best_col in zip(duplicate_indices_tuples, best_cols):
        for dup in indices:
            if dup != best_col and dup in updated_elites:
                updated_elites[updated_elites.index(dup)] = best_col

    # Drop the non-best duplicates
    to_drop = [idx for grp in duplicate_indices_tuples for idx in grp if idx not in best_cols]
    new_configs = configs.drop(index=to_drop)
    new_training_exps = training_exps.drop(columns=to_drop)

    return new_configs, new_training_exps, updated_elites


def calc_dev(full_exps, dev):
    for idx, row in full_exps.iterrows():
        min_value = row.min()
        dev.loc[idx] = row.apply(
            lambda x, minimum_value=min_value: (
                100 * (x - minimum_value) / ((x + minimum_value) / 2)
                if pd.notna(x)
                else x
            )
        )


irace = importr("irace")


def load_irace_elite_configs(file_location):
    ro.r["load"](file_location)
    objects = ro.r["iraceResults"]
    testing = ListVector(objects.rx2("testing"))
    test_exp_matrix = testing.rx2("experiments")
    test_matrix_row_names = list(test_exp_matrix.rownames)
    test_matrix_col_names = list(test_exp_matrix.colnames)
    training_exps = objects.rx2("experiments")
    training_exps_col_names = list(training_exps.colnames)
    with (ro.default_converter + pandas2ri.converter).context():
        instances = irace.get_instanceID_seed_pairs(objects, instances=True)
        test_exp_matrix = pandas2ri.rpy2py(test_exp_matrix)
        test_exp_matrix = pd.DataFrame(
            test_exp_matrix, index=test_matrix_row_names, columns=test_matrix_col_names
        )
        training_exps = pandas2ri.rpy2py(training_exps)
        training_exps = pd.DataFrame(
            training_exps,
            index=instances["instanceID"],
            columns=training_exps_col_names,
        )
        training_exps = training_exps.sort_index()
        configs = objects.rx2("allConfigurations")
        configs = configs.replace(-2147483648, None)
        for col in configs.columns:
            configs[col] = configs[col].apply(
                lambda x: None if isinstance(x, SEXP.NACharacterType) else x
            )
        configs = configs.drop(columns=[".ID.", ".PARENT."])
        configs = configs.rename(columns=parameter_translation_map)
        for col in configs.select_dtypes(include=["float"]):
            if (configs[col] % 1 == 0).all():
                configs[col] = configs[col].astype("Int64")
        for col in configs.columns:
            if col in distribution_cols:
                configs[col] = configs[col].map(distribution_translation_map)
            if col in tree_traversal_cols:
                configs[col] = configs[col].map(tree_traversal_translation_map)
            if col == "RT":
                configs[col] = configs[col].map(reservation_type_translation_map)
        elites = objects.rx2("allElites")[-1]
        elites = [str(e) for e in elites]
        configs, training_exps, elites = remove_duplicates(
            configs, training_exps, test_exp_matrix, elites
        )
        elite_configs = configs.loc[elites]
        elite_configs = elite_configs.dropna(axis=1, how="all")
        for col in elite_configs.select_dtypes(include=["float"]):
            if (elite_configs[col] % 1 == 0).all():
                elite_configs[col] = elite_configs[col].astype("Int64")

        all_exps = pd.concat([training_exps, test_exp_matrix], axis=0)
        # coming from all_exps, for every row, in each cell change the value for the deviation from the best config
        dev = all_exps.copy()
        calc_dev(all_exps, dev)
        return configs, elite_configs, test_exp_matrix, training_exps, all_exps, dev


configs_map = {}
elite_configs_map = {}
testing_exps_map = {}
training_exps_map = {}
full_experiments_map = {}
deviation_map = {}
for root, _, files in os.walk("irace_results"):
    for file in files:
        if file.endswith("irace.Rdata"):
            file_path = os.path.join(root, file)
            if "A" in file_path or "B" in file_path:
                continue
            dir_path = os.path.dirname(file_path)
            splitted = dir_path.split(os.sep)
            budget = splitted[1].split("_")[0]
            instance_type = splitted[2]
            server_capacity = splitted[3]
            try:
                configs, elite_configs, testing_exps, full_exps, all_exps, dev = (
                    load_irace_elite_configs(file_path)
                )
                # Ensure 'small' comes before 'medium' in the map
                if instance_type not in configs_map:
                    configs_map[instance_type] = {}
                configs_map[instance_type][server_capacity] = configs
                if instance_type not in elite_configs_map:
                    elite_configs_map[instance_type] = {}
                elite_configs_map[instance_type][server_capacity] = elite_configs
                if instance_type not in testing_exps_map:
                    testing_exps_map[instance_type] = {}
                testing_exps_map[instance_type][server_capacity] = testing_exps
                if instance_type not in training_exps_map:
                    training_exps_map[instance_type] = {}
                training_exps_map[instance_type][server_capacity] = full_exps
                if instance_type not in full_experiments_map:
                    full_experiments_map[instance_type] = {}
                full_experiments_map[instance_type][server_capacity] = all_exps
                if instance_type not in deviation_map:
                    deviation_map[instance_type] = {}
                deviation_map[instance_type][server_capacity] = dev
            except Exception as e:
                print(f"Failed to process {file_path}: {e}")


# Reorder the maps so 'small' comes first, then 'medium', and server capacities are sorted numerically
def reorder_instance_types_and_capacities(d):
    ordered = {}
    for instance in ["small", "medium"]:
        if instance in d:
            # Sort server capacities numerically
            capacities = d[instance]
            ordered_capacities = dict(
                sorted(capacities.items(), key=lambda x: int(x[0]))
            )
            ordered[instance] = ordered_capacities
    for instance in d:
        if instance not in ordered:
            capacities = d[instance]
            ordered_capacities = dict(
                sorted(capacities.items(), key=lambda x: int(x[0]))
            )
            ordered[instance] = ordered_capacities
    return ordered


configs_map = reorder_instance_types_and_capacities(configs_map)
elite_configs_map = reorder_instance_types_and_capacities(elite_configs_map)
testing_exps_map = reorder_instance_types_and_capacities(testing_exps_map)
training_exps_map = reorder_instance_types_and_capacities(training_exps_map)
full_experiments_map = reorder_instance_types_and_capacities(full_experiments_map)
deviation_map = reorder_instance_types_and_capacities(deviation_map)

In [4]:
def config_summary_str(config_row, subindex=None):
    alg = config_row["algorithm"]
    if subindex is not None:
        result = f"{alg}<sub>{subindex}</sub>("
    else:
        result = f"{alg}("
    for col in config_row.index:
        if col == "algorithm":
            continue
        value = config_row[col]
        if pd.isna(value):
            continue
        if isinstance(value, (float, np.floating)):
            value = int(value)
        key = col
        if "MR" in col:
            key = "MR"
        result += f"{key}={value}, "
    return result[:-2] + ")"

summaries_elites = {}
for instance_type, server_capacities in full_experiments_map.items():

    for server_capacity, full_exps in server_capacities.items():
        elite_configs = elite_configs_map[instance_type][server_capacity]
        elite_index = elite_configs.index
        devs = deviation_map[instance_type][server_capacity]
        # mean deviation
        # For most operations, A is skipped for server_capacity 50 as there isn't enough data to get accurate results, as algorithm A fails on most cases of server capacity 50
        mean_devs = []
        for idx in elite_index:
            mean_devs.append(devs[idx].mean())
        mean_devs = np.array(mean_devs)

        # number of instances that config is run on
        # n_instances = [full_exps[idx].count() for idx in elite_index]

        # number of instances where deviation is 0 for an idx
        n_best = [0] * len(elite_index)
        best_devs = devs[elite_index]
        for row_idx, row in best_devs.iterrows():
            # get the column with the minimum value
            col_name = row.idxmin()
            # get the position of col_name in elite_index (a pandas index)
            idx = elite_index.get_loc(col_name)
            n_best[idx] += 1

        # mean rank
        mean_rank = [0] * len(elite_index)
        best_exps = full_exps[elite_index]
        for row_idx, row in best_exps.iterrows():
            ranks = row.rank(method="min")
            for i, idx in enumerate(elite_index):
                rank = ranks[idx]
                if pd.notna(rank):
                    mean_rank[i] += rank
        mean_rank = np.array(mean_rank) / len(best_exps)

        configs = configs_map[instance_type][server_capacity]
        summary_config = [
            config_summary_str(configs.loc[idx], subindex=i + 1)
            for i, idx in enumerate(elite_index)
        ]
        if instance_type not in summaries_elites:
            summaries_elites[instance_type] = {}
        summaries_elites[instance_type][server_capacity] = {
            "Configuration": summary_config,
            "Config ID": elite_index,
            # "# Instances run on": n_instances,
            "Mean RPD": mean_devs,
            "# Best": n_best,
            "Mean Rank": mean_rank,
        }

display("Summary:")
summary = pd.DataFrame(
    columns=[
        "Instance type",
        "Server capacity",
        "Config ID",
        "Configuration",
        # "# Instances run on",
        "Mean RPD",
        "# Best",
        "Mean Rank",
    ]
)
for instance_type, server_capacities in summaries_elites.items():
    for server_capacity, summary_data in server_capacities.items():
        n = len(summary_data["Configuration"])
        df = pd.DataFrame(
            {
                "Instance type": [instance_type] * n,
                "Server capacity": [server_capacity] * n,
                **summary_data,
            }
        )
        summary = pd.concat([summary, df], ignore_index=True)
display(summary.style.hide(axis="index"))

'Summary:'

C:\Users\millenium\AppData\Local\Temp\ipykernel_24496\1162529185.py:98: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  summary = pd.concat([summary, df], ignore_index=True)


Instance type,Server capacity,Config ID,Configuration,Mean RPD,# Best,Mean Rank
small,50,2094,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, OVSP=LL, NVSP=HL, PSP=RR)",2.875588,10,2.800000
small,50,3304,"C2(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, OVSP=RND, NVSP=HL, PSP=RND)",2.838135,10,2.500000
small,50,4037,"C3(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, OVSP=RND, NVSP=LL, PSP=RND)",2.838135,0,2.500000
small,50,7035,"C4(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, OVSP=RND, NVSP=HL, PSP=RR)",2.862590,10,3.100000
small,50,11581,"C5(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, OVSP=HL, NVSP=HL, PSP=RR)",2.757602,10,2.925000
small,150,13764,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, OVSP=RND, NVSP=LL, PSP=RR)",2.534669,7,3.000000
small,150,11479,"C2(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, OVSP=RR, NVSP=HL, PSP=RR)",2.521540,13,2.675000
small,150,10236,"C3(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, OVSP=RND, NVSP=LL, PSP=RR)",2.672769,10,3.275000
small,150,4085,"C4(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, OVSP=RND, NVSP=LL, PSP=LL)",2.641216,10,2.525000
small,150,5220,"C5(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, OVSP=RND, NVSP=HL, PSP=LL)",2.641216,0,2.525000


In [5]:
# The first elite config is the best one considered by irace, so we run each best elite config on all combinations of instance types and server capacities
# (running them only on test instances)
best_elite_runs = pd.DataFrame(
    columns=[
        "Instance type",
        "Server capacity",
        "Was best elite",
        "Instance",
        "Configuration",
        "Cost",
        # "Time"
    ]
)

best_elites = {
    "C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, OVSP=LL, NVSP=HL, PSP=RR)": [
        ("small", "50"),
        ("medium", "50"),
    ],
    "C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, OVSP=RND, NVSP=LL, PSP=RR)": (
        "small",
        "150",
    ),
    "C1(MR=2, RT=SES, EVST=TTB, EVSP=LL, OVST=TTB, OVSP=LL, NVSP=LL, PSP=LL)": (
        "small",
        "650",
    ),
    "C1(MR=9, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, OVSP=LL, NVSP=HL, PSP=LL)": (
        "small",
        "1000",
    ),
    "C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, OVSP=HL, NVSP=LL, PSP=RR)": (
        "medium",
        "150",
    ),
    "C1(MR=6, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, OVSP=LL, NVSP=HL, PSP=RND)": (
        "medium",
        "650",
    ),
    "C1(MR=9, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, OVSP=LL, NVSP=LL, PSP=LL)": (
        "medium",
        "1000",
    ),
    "C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, OVSP=RR, NVSP=LL, PSP=RND)": (
        "big",
        "50",
    ),
    "C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, OVSP=RR, NVSP=LL, PSP=LL)": (
        "big",
        "150",
    ),
    "C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, OVSP=RR, NVSP=HL, PSP=RR)": (
        "big",
        "650",
    ),
    "C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, OVSP=RND, NVSP=HL, PSP=RND)": (
        "big",
        "1000",
    )
}


def parse_log(file, instance_type):
    with open(file, "r") as f:
        df_row = {
            "Instance type": instance_type,
            "Server capacity": None,
            "Was best elite": None,
            "Configuration": None,
            "Instance": None,
            "Cost": None,
        }
        for line in f:
            if "Output for" in line:
                # Find substring CAPACITY= and read the number after it, preceeding the ,
                server_capacity = line.split("CAPACITY=")[1].split(",")[0]
                instance = line.split(",")[1].split("=")[1]
                real_instance = str(int(instance) - 29) + "t"
                config = line.split("args=")[1].strip()
                best_config = best_elites.get(config, None)
                if isinstance(best_config, list):
                    for best in best_config:
                        df_row["Was best elite"] = "F"
                        if best[0] == instance_type and best[1] == server_capacity:
                            df_row["Was best elite"] = "T"
                            break
                else:
                    if (
                        best_config[0] == instance_type
                        and best_config[1] == server_capacity
                    ):
                        df_row["Was best elite"] = "T"
                    else:
                        df_row["Was best elite"] = "F"
                df_row["Server capacity"] = int(server_capacity)
                df_row["Instance"] = real_instance
                df_row["Configuration"] = config
            else:
                cost = float(line.split(" ")[0])
                df_row["Cost"] = cost
                best_elite_runs.loc[len(best_elite_runs)] = df_row


parse_log("test_elite_configs/small.log", "small")
parse_log("test_elite_configs/medium.log", "medium")
parse_log("test_elite_configs/big.log", "big")
display(
    best_elite_runs.sort_values(
        by=["Instance type", "Server capacity", "Instance"],
        ascending=[False, True, True],
    )
)

,Instance type,Server capacity,Was best elite,Instance,Configuration,Cost
9,small,50,T,10t,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",3.292536e+06
49,small,50,F,10t,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, O...",3.234938e+06
89,small,50,F,10t,"C1(MR=2, RT=SES, EVST=TTB, EVSP=LL, OVST=TTB, ...",3.290405e+06
129,small,50,F,10t,"C1(MR=9, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, O...",3.580695e+06
169,small,50,F,10t,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",3.428797e+06
...,...,...,...,...,...,...
1158,big,1000,F,9t,"C1(MR=9, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",2.190575e+07
1198,big,1000,F,9t,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",2.223317e+07
1238,big,1000,F,9t,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",2.243562e+07
1278,big,1000,F,9t,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",2.181769e+07


In [6]:
# For each combination of instance type, server capacity and instance in best_elite_runs, calculate the RPD of all configurations
instance_types = best_elite_runs["Instance type"].unique()
server_capacities = best_elite_runs["Server capacity"].unique()
instances = best_elite_runs["Instance"].unique()
for instance_type in instance_types:
    for server_capacity in server_capacities:
        for instance in instances:
            best_elites = best_elite_runs[
                (best_elite_runs["Instance type"] == instance_type)
                & (best_elite_runs["Server capacity"] == server_capacity)
                & (best_elite_runs["Instance"] == instance)
            ]
            ranks = best_elites["Cost"].rank(method="min")
            best_elite_runs.loc[
                (best_elite_runs["Instance type"] == instance_type)
                & (best_elite_runs["Server capacity"] == server_capacity)
                & (best_elite_runs["Instance"] == instance),
                "Rank",
            ] = ranks
            best_elite_run = best_elites.sort_values(by="Cost").iloc[0]
            best_elite_config = best_elite_run["Configuration"]
            best_elite_cost = best_elite_run["Cost"]
            for idx_row, row in best_elites.iterrows():
                config = row["Configuration"]
                cost = row["Cost"]
                rpd = 100 * abs((cost - best_elite_cost) / best_elite_cost)
                best_elite_runs.loc[
                    (best_elite_runs["Instance type"] == instance_type)
                    & (best_elite_runs["Server capacity"] == server_capacity)
                    & (best_elite_runs["Instance"] == instance)
                    & (best_elite_runs["Configuration"] == config)
                    & (best_elite_runs["Cost"] == cost),
                    "RPD",
                ] = rpd.round(3)
                best_elite_runs.loc[
                    (best_elite_runs["Instance type"] == instance_type)
                    & (best_elite_runs["Server capacity"] == server_capacity)
                    & (best_elite_runs["Instance"] == instance)
                    & (best_elite_runs["Configuration"] == config)
                    & (best_elite_runs["Cost"] == cost),
                    "Is best",
                ] = (
                    cost == best_elite_cost
                )

display(
    best_elite_runs.sort_values(
        by=["Instance type", "Server capacity", "Instance"],
        ascending=[False, True, True],
    )
)

C:\Users\millenium\AppData\Local\Temp\ipykernel_24496\2691363576.py:35: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'False' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  best_elite_runs.loc[


,Instance type,Server capacity,Was best elite,Instance,Configuration,Cost,Rank,RPD,Is best
9,small,50,T,10t,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",3.292536e+06,5.0,1.780,False
49,small,50,F,10t,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, O...",3.234938e+06,1.0,0.000,True
89,small,50,F,10t,"C1(MR=2, RT=SES, EVST=TTB, EVSP=LL, OVST=TTB, ...",3.290405e+06,4.0,1.715,False
129,small,50,F,10t,"C1(MR=9, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, O...",3.580695e+06,10.0,10.688,False
169,small,50,F,10t,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",3.428797e+06,9.0,5.993,False
...,...,...,...,...,...,...,...,...,...
1158,big,1000,F,9t,"C1(MR=9, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",2.190575e+07,6.0,1.819,False
1198,big,1000,F,9t,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",2.223317e+07,8.0,3.341,False
1238,big,1000,F,9t,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",2.243562e+07,10.0,4.282,False
1278,big,1000,F,9t,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",2.181769e+07,5.0,1.410,False


In [7]:
def group_alt_results(group_array, sort_by=None, ascending=None):
    avgs = best_elite_runs.groupby(group_array)["RPD"].mean().round(3).reset_index()
    avgs = avgs.rename(columns={"RPD": "Mean RPD"})

    max_rpd = best_elite_runs.groupby(group_array)["RPD"].max().round(3).reset_index()
    avgs = avgs.merge(
        max_rpd,
        on=group_array,
    )
    avgs = avgs.rename(columns={"RPD": "Max RPD"})

    q1 = (
        best_elite_runs.groupby(group_array)["RPD"]
        .quantile(0.25)
        .round(3)
        .reset_index()
    )
    avgs = avgs.merge(q1, on=group_array)
    avgs = avgs.rename(columns={"RPD": "Q1 RPD"})
    median = best_elite_runs.groupby(group_array)["RPD"].median().round(3).reset_index()
    avgs = avgs.merge(
        median,
        on=group_array,
    )
    avgs = avgs.rename(columns={"RPD": "Median RPD"})
    q3 = (
        best_elite_runs.groupby(group_array)["RPD"]
        .quantile(0.75)
        .round(3)
        .reset_index()
    )
    avgs = avgs.merge(q3, on=group_array)
    avgs = avgs.rename(columns={"RPD": "Q3 RPD"})

    best_counts = (
        best_elite_runs.groupby(group_array)["Is best"]
        .apply(lambda x: x.eq(True).sum())
        .reset_index(name="# Best")
    )
    avgs = avgs.merge(
        best_counts,
        on=group_array,
    )
    avgs["Mean Rank"] = (
        best_elite_runs.groupby(group_array)["Rank"]
        .mean()
        .round(3)
        .reset_index(drop=True)
    )

    return avgs.sort_values(by=sort_by, ascending=ascending)


# convert to tex table
def to_latex_table(df, filename):
    with open(filename, "w") as f:
        f.write(
            df.to_latex(
                index=False,
                float_format="%.3f",
                column_format="l" + "c" * (len(df.columns) - 1),
                escape=False,
            )
        )

avgs = group_alt_results(
    ["Instance type", "Server capacity", "Was best elite", "Configuration"],
    sort_by=["Instance type", "Server capacity", "Configuration"],
    ascending=[False, True, True],
)
display(avgs)
to_latex_table(avgs, filename="alt_results.tex")

,Instance type,Server capacity,Was best elite,Configuration,Mean RPD,Max RPD,Q1 RPD,Median RPD,Q3 RPD,# Best,Mean Rank
98,small,50,T,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",3.792,25.059,0.303,1.211,2.378,0,4.1
88,small,50,F,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",3.868,26.078,0.319,0.555,2.836,1,4.4
89,small,50,F,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",3.636,25.537,0.000,0.000,2.704,6,3.0
90,small,50,F,"C1(MR=2, RT=SES, EVST=TTB, EVSP=LL, OVST=TTB, ...",4.730,11.340,2.132,3.919,5.477,1,5.8
91,small,50,F,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",7.469,33.552,3.363,5.361,6.010,0,8.2
...,...,...,...,...,...,...,...,...,...,...,...
43,big,1000,T,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",0.531,1.257,0.167,0.414,0.955,2,3.4
39,big,1000,F,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",0.684,1.561,0.268,0.602,1.083,1,4.0
40,big,1000,F,"C1(MR=6, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",0.988,1.777,0.460,1.040,1.568,1,4.8
41,big,1000,F,"C1(MR=9, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, O...",2.481,4.979,1.550,2.046,3.382,0,8.3


In [8]:
avgs_server = group_alt_results(
    ["Server capacity", "Configuration"],
    sort_by=["Server capacity", "Configuration"],
    ascending=[True, True],
)

display(avgs_server)
to_latex_table(avgs_server, filename="alt_results_server.tex")

,Server capacity,Configuration,Mean RPD,Max RPD,Q1 RPD,Median RPD,Q3 RPD,# Best,Mean Rank
0,50,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",1.531,25.059,0.036,0.244,1.041,6,3.233
1,50,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",1.544,26.078,0.082,0.253,0.738,4,3.367
2,50,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",1.485,25.537,0.000,0.058,0.905,10,3.000
3,50,"C1(MR=2, RT=SES, EVST=TTB, EVSP=LL, OVST=TTB, ...",11.629,25.236,4.897,11.227,17.289,1,8.000
4,50,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",3.518,33.552,0.827,2.140,3.986,5,5.367
5,50,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, O...",1.877,3.714,1.238,1.997,2.641,2,4.333
6,50,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",2.056,4.830,1.132,1.939,2.640,1,5.400
7,50,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",1.991,4.536,1.311,1.964,2.464,1,4.867
8,50,"C1(MR=6, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",8.890,12.350,7.557,8.939,10.831,0,8.067
9,50,"C1(MR=9, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, O...",17.790,23.985,16.251,17.937,20.934,0,10.267


In [9]:
avgs_instance = group_alt_results(
    ["Instance type", "Configuration"],
    sort_by=["Instance type", "Configuration"],
    ascending=[False, True],
)

display(avgs_instance)
to_latex_table(avgs_instance, filename="alt_results_instance.tex")

,Instance type,Configuration,Mean RPD,Max RPD,Q1 RPD,Median RPD,Q3 RPD,# Best,Mean Rank
22,small,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",6.183,25.059,1.244,3.715,9.488,2,6.550
23,small,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",5.969,26.078,1.083,3.812,9.046,3,6.375
24,small,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",6.725,25.537,1.414,5.138,11.079,7,6.525
25,small,"C1(MR=2, RT=SES, EVST=TTB, EVSP=LL, OVST=TTB, ...",3.370,11.340,1.454,2.548,4.630,4,4.550
26,small,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",6.545,33.552,3.234,5.226,7.352,1,7.675
27,small,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, O...",4.523,23.230,1.715,3.540,6.516,2,5.500
28,small,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",3.937,14.225,1.488,3.429,5.520,4,5.300
29,small,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",4.579,17.298,1.704,3.366,6.270,0,5.825
30,small,"C1(MR=6, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",3.909,12.350,1.286,3.271,6.721,5,4.975
31,small,"C1(MR=9, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, O...",6.263,19.627,1.949,4.822,10.314,5,6.550


In [10]:
avgs_all = group_alt_results(
    ["Configuration"],
    sort_by=["Configuration"],
    ascending=[True],
)
display(avgs_all)
to_latex_table(avgs_all, filename="alt_results_all.tex")

,Configuration,Mean RPD,Max RPD,Q1 RPD,Median RPD,Q3 RPD,# Best,Mean Rank
0,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",3.894,25.059,0.511,1.792,6.296,10,6.067
1,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",3.844,26.078,0.326,1.772,6.217,12,5.958
2,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",4.131,25.537,0.436,1.930,6.342,14,6.258
3,"C1(MR=2, RT=SES, EVST=TTB, EVSP=LL, OVST=TTB, ...",6.340,25.236,2.429,4.214,8.120,4,7.475
4,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",3.231,33.552,0.352,2.117,4.415,25,5.317
5,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, O...",2.609,23.230,0.756,1.785,3.386,5,4.900
6,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",2.455,14.225,0.936,1.672,3.340,9,5.083
7,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",2.688,17.298,0.964,1.644,3.306,3,5.183
8,"C1(MR=6, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",3.533,12.350,0.517,2.027,6.715,21,5.267
9,"C1(MR=9, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, O...",6.832,23.985,1.750,4.386,10.314,5,7.550


In [ ]:
def run_stat_tests(
    group_label_1,
    group_label_2,
    friedmann_tests,
    wilcoxon_tests,
    full_exps,
    best_config_ids,
):
    common_indices = full_exps.index
    for config_id in best_config_ids:
        common_indices = common_indices.intersection(
            full_exps[config_id].dropna().index
        )
    if common_indices.empty:
        return
    friedmann_samples = [
        full_exps[config_id].loc[common_indices] for config_id in best_config_ids
    ]

    stat, p_value = friedmanchisquare(*friedmann_samples)

    friedmann_tests.loc[len(friedmann_tests)] = {
        "Instance Type": group_label_1,
        "Server Capacity": group_label_2,
        "statistic": stat,
        "p-value": p_value,
    }

    labels = {config_id: "" for config_id in best_config_ids}
    labels[best_config_ids[0]] = "Best A"
    labels[best_config_ids[1]] = "Best B"
    for i, config_id in enumerate(best_config_ids[2:]):
        labels[config_id] = f"Elite config {i+1}"

    for config1, config2 in combinations(best_config_ids, 2):

        common_indices = (
            full_exps[config1]
            .dropna()
            .index.intersection(full_exps[config2].dropna().index)
        )
        config1_sample = full_exps[config1].loc[common_indices]
        config2_sample = full_exps[config2].loc[common_indices]

        diff = config1_sample - config2_sample


        # Does the medians of the differences differ?


        # If the p value is less than 0.05, then config2 is significantly better than config1
        if not (diff == 0).all():

            stat, p_value = wilcoxon(diff.round(3), alternative="greater")

            wilcoxon_tests.loc[len(wilcoxon_tests)] = {
                "Instance Type": group_label_1,
                "Server Capacity": group_label_2,
                "Config 1": labels[config1],
                "Config 2": labels[config2],
                "W": stat,
                "p-value": p_value,
            }



friedmann_tests = pd.DataFrame(
    columns=["Instance Type", "Server Capacity", "statistic", "p-value"]
)


wilcoxon_tests = pd.DataFrame(
    columns=[
        "Instance Type",
        "Server Capacity",
        "Config 1",
        "Config 2",
        "W",
        "p-value",
    ]
)



for instance_type, server_capacities in full_experiments_map.items():

    for server_capacity, full_exps in server_capacities.items():

        run_stat_tests(
            instance_type,
            server_capacity,
            friedmann_tests,
            wilcoxon_tests,
            full_exps,
            best_config_ids_map[instance_type][server_capacity],
        )



display("Friedmann Tests Results")


display(friedmann_tests.style.hide(axis="index"))


friedmann_tests.to_csv("friedmann_tests_results.csv", index=False)


display("Wilcoxon Tests Results")


display(wilcoxon_tests.style.hide(axis="index"))


wilcoxon_tests.to_csv("wilcoxon_tests_results.csv", index=False)

In [ ]:
merged_map = {}


def calc_deviations(instance_type, server_capacity, merged):
    deviation_df = merged.copy()
    deviation_df["min_objective_per_instance"] = deviation_df.groupby("instance")[
        "objective"
    ].transform("min")
    # Use percentage deviation
    deviation_df["RPD"] = 100 * abs(
        (deviation_df["objective"] - deviation_df["min_objective_per_instance"])
        / deviation_df["min_objective_per_instance"]
    )

    mean_deviation_per_instance = (
        deviation_df.groupby("instance")["RPD"].mean().sort_index()
    )

    elite_configs = elite_configs_map[instance_type][server_capacity]
    elite_merged = merged[merged["config_id"].isin(elite_configs.index.astype(str))]
    elite_deviation_df = elite_merged.copy()
    elite_deviation_df["min_objective_per_instance"] = deviation_df.groupby("instance")[
        "objective"
    ].transform("min")
    elite_deviation_df["RPD"] = 100 * abs(
        (
            elite_deviation_df["objective"]
            - elite_deviation_df["min_objective_per_instance"]
        )
        / elite_deviation_df["min_objective_per_instance"]
    )
    return mean_deviation_per_instance, elite_deviation_df


def summarize_deviations(mean_deviation_all, elite_deviation_all):
    # Calculate summary statistics for mean deviation per instance (all configs)
    summary_df = pd.DataFrame(
        {
            "Avg Mean Deviation (%) (All)": [mean_deviation_all.mean()],
        }
    )
    summary_df.index = ["All Configs"]

    elite_config_ids = elite_deviation_all["config_id"].unique()
    # Calculate summary statistics for each elite config individually
    elite_config_stats = []
    for i in range(len(elite_config_ids)):
        config_id = elite_config_ids[i]
        df_all = elite_deviation_all[elite_deviation_all["config_id"] == config_id]
        all_per_instance = df_all.groupby("instance")["RPD"].mean().sort_index()

        elite_config_stats.append(
            {
                "config_id": "Elite Config " + str(i + 1),
                "Avg Mean Deviation (%) (All)": all_per_instance.mean(),
            }
        )
    if elite_config_stats:
        elite_config_stats_df = pd.DataFrame(elite_config_stats).set_index("config_id")
        elite_config_stats_df.index.name = "Elite Config ID"
    # Round and display all together
    display(
        f"Mean Deviation (%) Summary Statistics for instance type {instance_type} and server capacity {server_capacity}:"
    )
    display(pd.concat([summary_df, elite_config_stats_df]).round(3))


for instance_type, server_capacities in full_experiments_map.items():
    for server_capacity, full_exps in server_capacities.items():
        configs = configs_map[instance_type][server_capacity]
        # only keep the columns that are in configs
        c_configs = configs[configs["algorithm"] == "C"].dropna(axis=1, how="all")
        c_configs = c_configs.drop(columns=["algorithm"]).rename(
            columns={"MR [C]": "MR"}
        )
        e = full_exps[c_configs.index]

        # Prepare data for ANOVA: melt experiments to long format and merge with parameter values
        exp_long = e.reset_index().melt(
            id_vars="index", var_name="config_id", value_name="objective"
        )
        exp_long = exp_long.rename(columns={"index": "instance"})
        exp_long["config_id"] = exp_long["config_id"].astype(str)
        c_configs_ = c_configs.copy()
        c_configs_["config_id"] = c_configs_.index.astype(str)
        merged = exp_long.merge(c_configs_, on="config_id")
        merged["MR"] = merged["MR"].astype(int)

        mean_deviation_per_instance, elite_deviation_df = calc_deviations(
            instance_type, server_capacity, merged
        )

        summarize_deviations(
            mean_deviation_per_instance,
            elite_deviation_df,
        )

        if instance_type not in merged_map:
            merged_map[instance_type] = {}
        merged_map[instance_type][server_capacity] = merged

In [ ]:
for instance_type, server_capacities in training_exps_map.items():
    for server_capacity, full_exps in server_capacities.items():
        merged = merged_map[instance_type][server_capacity]
        df_clean = merged.dropna(subset=["objective"]).copy()

        # Get elite config ids for this instance/capacity
        elite_configs = elite_configs_map[instance_type][server_capacity]
        elite_configs = elite_configs.rename(columns={"MR [C]": "MR"})
        elite_ids = set(elite_configs.index.astype(str))

        # Compute deviation from the best for each instance
        df_clean.loc[:, "min_objective_per_instance"] = df_clean.groupby("instance")[
            "objective"
        ].transform("min")
        df_clean.loc[:, "Mean RPD"] = 100 * abs(
            (df_clean["objective"] - df_clean["min_objective_per_instance"])
            / df_clean["min_objective_per_instance"]
        )

        # Compute and display mean deviation from best for each parameter in the data
        param_cols = df_clean.columns.difference(
            [
                "instance",
                "config_id",
                "objective",
                "min_objective_per_instance",
                "Mean RPD",
            ]
        )
        for param in param_cols:
            mean_deviation_param = (
                df_clean.groupby(param)["Mean RPD"].mean().sort_values()
            )

            # For each parameter value, find elite config ids that chose it
            elite_param_map = {}
            for value in mean_deviation_param.index:
                elite_ids_with_value = (
                    elite_configs[elite_configs[param] == value]
                    .index.astype(str)
                    .tolist()
                )
                elite_param_map[value] = (
                    elite_ids_with_value if elite_ids_with_value else None
                )

            # Create a DataFrame for display
            result_df = mean_deviation_param.round(3).to_frame("Mean RPD")
            result_df["In Elite"] = result_df.index.map(lambda v: elite_param_map[v])

            display(
                f"Mean RPD for {instance_type}, Server Capacity {server_capacity} by {param}:"
            )
            display(result_df)

In [ ]:
# Scatterplot: MR vs objective, colored by RT
for instance_type, server_capacities in training_exps_map.items():
    for server_capacity, full_exps in server_capacities.items():
        merged = merged_map[instance_type][server_capacity]
        for col in merged.columns:
            if col not in ["instance", "objective", "config_id"]:
                fig, axs = plt.subplots(figsize=(24, 30), ncols=3, nrows=10)
                unique_instances = merged["instance"].unique()
                axs_flat = axs.flatten()
                fig.suptitle(f"Scatterplot of {col} vs Objective", y=0.98)
                for i, instance in enumerate(unique_instances):
                    ax = axs_flat[i]
                    sns.scatterplot(
                        data=merged[merged["instance"] == instance],
                        x=col,
                        y="objective",
                        ax=ax,
                    )
                    ax.set_title(
                        f"Instance {instance} ({instance_type}, Server Capacity {server_capacity})"
                    )
                    ax.set_xlabel(col)
                    ax.set_ylabel("Objective")
                # Hide unused axes
                for j in range(len(unique_instances), len(axs_flat)):
                    axs_flat[j].set_visible(False)
                plt.tight_layout(rect=[0, 0, 1, 0.96])
                plt.subplots_adjust(top=0.93)
        break

In [ ]:
# testing_exp is a DataFrame where each column represents a parameter configuration, each row an instance.
# The cell contains the value of the objective function for that configuration ran on that instance.
def calculate_statistics(testing_exp):
    # Calculate mean, std, and 95% CI for each configuration (column), taking into account all instances (rows)
    means = testing_exp.mean(axis=0)
    stds = testing_exp.std(axis=0, ddof=1)
    n = testing_exp.shape[0]
    ci95 = 1.96 * stds / (n**0.5)

    stats = pd.DataFrame({"Mean": means, "Std Dev": stds, "95% CI": ci95})

    # Sort by mean (assuming lower is better; change to ascending=False if higher is better)
    stats = stats.sort_values("Mean")
    return stats


for instance_type, server_capacities in testing_exps_map.items():
    for server_capacity, test_exp in server_capacities.items():
        stats = calculate_statistics(test_exp)
        display(f"Instance Type: {instance_type}, Server Capacity: {server_capacity}")
        display(stats)

In [ ]:
# calculate chosen parameter frequency
def calculate_chosen_parameter_frequency(configs):
    frequency = {}
    for col in configs.columns:
        if col == "algorithm" or col in alg_c_parameters:
            if col not in frequency:
                frequency[col] = {}
            for value in configs[col]:
                if value is None or (isinstance(value, float) and pd.isna(value)):
                    continue
                if isinstance(value, float) and value.is_integer():
                    value = int(value)
                if value not in frequency[col]:
                    frequency[col][value] = 0
                frequency[col][value] += 1
    return frequency


# for every instance type and server capacity, calculate the chosen parameter frequency
chosen_parameter_frequency_map = {}
for instance_type, server_capacities in configs_map.items():
    for server_capacity, config in server_capacities.items():
        if instance_type not in chosen_parameter_frequency_map:
            chosen_parameter_frequency_map[instance_type] = {}
        frequency = calculate_chosen_parameter_frequency(config)
        chosen_parameter_frequency_map[instance_type][server_capacity] = frequency
# for every instance type and server capacity, display the chosen parameter frequency
# Use subplots for every instance type and server capacity
for instance_type, server_capacities in chosen_parameter_frequency_map.items():
    for server_capacity, frequency in server_capacities.items():
        all_params = ["algorithm"] + sorted(
            [p for p in frequency.keys() if p != "algorithm"]
        )
        n_params = len(all_params)
        ncols = 3
        nrows = (n_params - 1 + ncols) // ncols
        fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3 * nrows))
        fig.suptitle(
            f"Parameter Frequency for {instance_type} - Server capacity {server_capacity}\n(Only parameters used by C algorithm are shown)",
        )
        axes = axes.flatten()
        for idx, param in enumerate(all_params):
            param_values = frequency[param]
            if param_values:
                freq_series = pd.Series(param_values)
                # Check if all keys are ints (or can be safely cast to int)
                try:
                    keys_as_int = [int(k) for k in freq_series.index]
                    is_all_int = all(
                        isinstance(k, int) or (isinstance(k, float) and k.is_integer())
                        for k in freq_series.index
                    )
                except Exception:
                    is_all_int = False
                    freq_series = freq_series.sort_index()
                if is_all_int and len(freq_series) > 1:
                    # Expand the series for histogram plotting
                    expanded = []
                    for k, v in freq_series.items():
                        expanded.extend([int(k)] * v)
                    axes[idx].hist(expanded, bins=20, edgecolor="black")
                    axes[idx].set_title(f"{param} histogram")
                    axes[idx].set_ylabel("Frequency")
                else:
                    sns.barplot(
                        x=freq_series.index.astype(str),
                        y=freq_series.values,
                        ax=axes[idx],
                        edgecolor="black",
                    )
                    axes[idx].set_title(f"{param} frequency")
                    axes[idx].set_xlabel("Parameter Value")
                    axes[idx].set_ylabel("Frequency")
            else:
                axes[idx].set_visible(False)
        for idx in range(len(all_params), len(axes)):
            axes[idx].set_visible(False)
        plt.tight_layout(rect=[0, 0, 1, 0.96])

In [ ]:
# heatmap of server capacity and max reservation in elite configs
heatmap_data = []
for instance_type, server_capacities in elite_configs_map.items():
    for server_capacity, elite_config in server_capacities.items():
        if "MR [C]" in elite_config.columns:
            heatmap_data.append(
                {
                    "Instance Type": instance_type,
                    "Server Capacity": server_capacity,
                    "Max Reservation": elite_config[elite_config["RT"] == "SER"][
                        "MR [C]"
                    ].mean(),
                }
            )
        if "MR" in elite_config.columns:
            heatmap_data.append(
                {
                    "Instance Type": instance_type,
                    "Server Capacity": server_capacity,
                    "Max Reservation": elite_config[elite_config["RT"] == "SER"][
                        "MR"
                    ].mean(),
                }
            )

heatmap_df = pd.DataFrame(heatmap_data)
heatmap_pivot = heatmap_df.pivot(
    index="Instance Type", columns="Server Capacity", values="Max Reservation"
)

# Sort columns numerically for correct order in heatmap
sorted_columns = sorted(heatmap_pivot.columns, key=lambda x: int(x))
heatmap_pivot = heatmap_pivot[sorted_columns]

plt.figure(figsize=(8, 4))
sns.heatmap(heatmap_pivot, annot=True, fmt=".2f", cmap="YlGnBu")
plt.title(
    "Heatmap of Average Max Reservation by Instance Type and Server Capacity\non Elite Configurations (Only Reservation Per Server)"
)
plt.ylabel("Instance Type")
plt.xlabel("Server Capacity")